# 4. Readings

Each translator writes every story of the three test sets in logic seven times: once by greedy decoding and six times
sampled at temperature 0.5. Decoding is constrained by a grammar to well-formed formulas over the story's vocabulary.
A configuration with *M* readings uses the first *M*.

The seed of reading *j* of the story at position *i* is `1000 i + j`, so a single story can be regenerated and gets
the readings it would get in a full run. Translator v2 is prompted with three solved stories; v3 and v4 without them.

The notebook also runs the controls outside FOLIO of Appendix C. Needs a GPU and the adapters of notebook 3. Run it
once per translator, restarting the runtime in between so that vLLM releases the GPU.

In [ ]:
# Copyright 2026 Javier Marín
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path

REPO = Path.cwd().parent
DATA = REPO / "data"
RUNS = REPO / "runs"      # everything this notebook produces goes here; data/ is never overwritten

import json

from logicverifier import data, models

TRANSLATOR = "v4"          # "v2", "v3" or "v4"

merged = models.merge_adapter(RUNS / "models" / "base", RUNS / "models" / f"{TRANSLATOR}_adapter",
                              RUNS / "models" / f"{TRANSLATOR}_merged")
llm = models.load_llm(merged)
sets = data.load_test_sets(DATA / "test_sets")

## Readings of the three test sets

In [ ]:
for name, stories in sets.items():
    readings = models.generate_readings(llm, list(stories.values()), few_shot=TRANSLATOR == "v2")
    path = RUNS / "readings" / TRANSLATOR / f"{name}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({"translator": TRANSLATOR, "set": name, "temperature": 0.5, "readings": readings},
                               ensure_ascii=False))
    stored = json.loads((DATA / "readings" / TRANSLATOR / f"{name}.json").read_text())["readings"]
    agree = sum(readings[i] == stored[i] for i in stored) / len(stored)
    print(f"{data.SETS[name]:24s} stories with the same seven readings as in the paper: {agree:.1%}")

Sampling on a GPU is not bit-for-bit reproducible across hardware and library versions, so a share of stories may get
different readings. Notebook 6 scores the stored readings; pointing it at `runs/readings/` scores these instead.

## Controls outside FOLIO

FOLIO without a vocabulary, the same DGT sentence in six languages, and common-sense questions in six languages.

In [ ]:
controls = models.evaluate_controls(llm, data.read_jsonl(RUNS / "folio" / "folio-validation.jsonl"),
                                    data.read_jsonl(DATA / "training" / "dgt_test.jsonl"),
                                    data.read_jsonl(DATA / "training" / "common_sense.jsonl"))
controls